# BookMatch: гибридная система

## 1. Признаки

Статистики и история рассчитаны по train. Тематические признаки получены из тегов.

In [1]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

from src.data import load_data, split_ratings
from src.features import HybridFeatures

books, ratings, tags, book_tags = load_data()
train, validation, test = split_ratings(ratings)
features = HybridFeatures().fit(train, books, tags, book_tags)
display(features.users.head())
display(features.books.head())

         user_mean  user_count  user_std
user_id                                 
1         4.000000           1   0.00000
2         5.000000           1   0.00000
3         1.000000           2   0.00000
4         5.000000           1   0.00000
5         4.666667           3   0.57735

         book_mean  book_count  book_std  tag_fiction  tag_i-own  tag_adult  \
book_id                                                                       
1         4.232323          99  0.890176            1          1          0   
2         4.210000         100  1.037625            1          1          0   
3         3.070707          99  1.465609            1          1          0   
4         4.460000         100  0.833879            1          1          1   
5         3.887755          98  1.082977            1          1          1   

         tag_novels  tag_own-it  tag_contemporary  tag_read-in-2015  ...  \
book_id                                                              ...   
1                 1           1                 1                 0  ...   
2                 1           1                 1                 1  ...   
3                 1           1                 1                 0  ...   
4                 1           1                 1                 

In [2]:
validation_features = features.transform(validation)
display(validation_features.head())
print('Пар:', len(validation_features))
print('Признаков:', validation_features.shape[1] - 2)
print('Пропусков:', validation_features.isna().sum().sum())

   user_id  book_id  user_mean  user_count  user_std  book_mean  book_count  \
0    30034     5534   3.302326          43  0.464701   3.416667          84   
1     5436      693   3.668508         181  1.174049   3.402062          97   
2    10111      967   4.021858         183  0.999760   3.707071          99   
3     7841     4385   3.375000          32  0.707107   4.043011          93   
4    13089     5385   3.444444          72  1.046643   3.125000          96   

   book_std  tag_fiction  tag_i-own  ...  tag_adult-fiction  tag_read-in-2013  \
0  0.996480            1          1  ...                  1                 1   
1  0.996450            1          1  ...                  1                 0   
2  0.823786            1          1  ...                  1                 0   
3  0.858639            1          0  ...                  1                 1   
4  1.068693            1          1  ...                  1                 0   

   tag_book-club  tag_audible  tag_fan

Пар: 45054
Признаков: 27
Пропусков: 0


## 2. Взвешенный гибрид

До 100 кандидатов от каждой модели. Оценки нормализованы в диапазон [0, 1]. Веса подбираются по nDCG@10 на 3 000 пользователях validation. Параметры базовых моделей взяты из первого проекта. Test не используется.

In [3]:
import json
import numpy as np
import pandas as pd
from threadpoolctl import threadpool_limits

from src.popularity import Popularity
from src.item_cf import ItemCF
from src.svd import SVDRecommender
from src.hybrid import Hybrid
from src.metrics import ranking_metrics

config = json.loads(Path('hybrid.json').read_text())
previous = pd.read_csv('reports/validation.csv')
params = {
    name: json.loads(group.loc[group['ndcg@10'].idxmax(), 'parameters'])
    for name, group in previous.groupby('model')
}
data = features.data
with threadpool_limits(limits=2):
    models = {
        'Popularity': Popularity(data, **params['Popularity']),
        'Content': features.content,
        'ItemCF': ItemCF(data).fit(**params['ItemCF']),
        'SVD': SVDRecommender(data).fit(train, seed=config['seed'], **params['SVD']),
    }
assert params['Content'] == {'max_tags': 30}
hybrid = Hybrid(models, config['weights'][0], config['candidates'])
relevant = validation.loc[validation.rating >= 4].set_index('user_id').book_id
rng = np.random.default_rng(config['seed'])
users = np.sort(rng.choice(np.sort(relevant.index), min(config['validation_users'], len(relevant)), replace=False))
pool = []
with threadpool_limits(limits=2):
    for start in range(0, len(users), 256):
        indices = data.users.get_indexer(users[start:start + 256])
        pool.extend(hybrid.get_candidates(indices))

candidate_hits = [relevant[user] in data.book_ids[items] for user, (items, _) in zip(users, pool)]
print('Доля попаданий в кандидаты:', round(np.mean(candidate_hits), 4))
print('Среднее число кандидатов:', round(np.mean([len(items) for items, _ in pool]), 1))

Доля попаданий в кандидаты: 0.5177
Среднее число кандидатов: 297.8


In [4]:
results = []
for weights in config['weights']:
    vector = np.array([weights[name] for name in models])
    rows = []
    for user, (items, values) in zip(users, pool):
        order = np.argsort(-(values @ vector), kind='stable')[:10]
        recommended = data.book_ids[items[order]]
        row = {}
        for k in [5, 10]:
            precision, recall, ndcg = ranking_metrics(recommended, {relevant[user]}, k)
            row.update({f'precision@{k}': precision, f'recall@{k}': recall, f'ndcg@{k}': ndcg})
        rows.append(row)
    results.append({'weights': json.dumps(weights), **pd.DataFrame(rows).mean().to_dict()})
results = pd.DataFrame(results)
results.to_csv('reports/hybrid_validation.csv', index=False)
display(results)
best = results.loc[results['ndcg@10'].idxmax()]
hybrid = Hybrid(models, json.loads(best.weights), config['candidates'])
user_id = int(users[0])
recommended = hybrid.get_recommendations(user_id, N=10)
assert len(recommended) == len(set(recommended)) == 10
assert not set(recommended) & set(train.loc[train.user_id == user_id, 'book_id'])
display(books.set_index('book_id').loc[recommended, ['title', 'authors']])

                                             weights  precision@5  recall@5  \
0  {"Popularity": 0.25, "Content": 0.25, "ItemCF"...     0.036133  0.180667   
1  {"Popularity": 0.0, "Content": 0.0, "ItemCF": ...     0.037933  0.189667   
2  {"Popularity": 0.0, "Content": 0.1, "ItemCF": ...     0.037667  0.188333   
3  {"Popularity": 0.0, "Content": 0.2, "ItemCF": ...     0.037600  0.188000   
4  {"Popularity": 0.0, "Content": 0.4, "ItemCF": ...     0.036800  0.184000   
5  {"Popularity": 0.05, "Content": 0.15, "ItemCF"...     0.037800  0.189000   
6  {"Popularity": 0.0, "Content": 0.15, "ItemCF":...     0.037600  0.188000   

     ndcg@5  precision@10  recall@10   ndcg@10  
0  0.136489      0.022633   0.226333  0.151069  
1  0.140883      0.024167   0.241667  0.157658  
2  0.140508      0.024500   0.245000  0.158610  
3  0.139819      0.024700   0.247000  0.158620  
4  0.136735      0.024333   0.243333  0.155802  
5  0.140517      0.024667   0.246667  0.158922  
6  0.140407      0.02473

                                                     title  \
book_id                                                      
9209                                           Roller Girl   
2565                                          Wonderstruck   
9381                                            Rain Reign   
8521                                                  Echo   
4323                                              El Deafo   
7305     The War that Saved My Life (The War That Saved...   
5476                                         The Crossover   
9229                               The Fourteenth Goldfish   
8059                                        Fish in a Tree   
9909                                                George   

                                          authors  
book_id                                            
9209                            Victoria Jamieson  
2565                               Brian Selznick  
9381                                Ann M. Mart

Лучшие веса: ItemCF 0,75, Content 0,15, SVD 0,10, Popularity 0. Кандидаты поступают от всех четырёх моделей.

nDCG@10: 0,159363 против 0,157658 у ItemCF (+1,1%). Попаданий в Top-10: 742 против 725. При равных весах nDCG@10 составляет 0,151069.

В пуле в среднем 297,8 книги. Покрытие релевантных книг: 51,77%. Преимущество гибрида пока проверено только на выборке подбора.